# VQE（変分量子固有値ソルバー）

量子回路で基底状態を近似的に作り、基底エネルギー$E_0$を求める方法。量子コンピュータで状態を作ってエネルギーを測り、古典コンピュータでパラメータを最適化する。

各模型への適用（アンザッツの選び方、測定のしかた、手計算）は、`models/<模型>/04_vqe.ipynb` にまとめてある。

## 1. 考え方

- **変分原理**

  任意の規格化された状態$\ket{\psi}$について

  \begin{equation}
  E(\psi) = \bra{\psi}H\ket{\psi} \ge E_0
  \end{equation}

  が成り立ち、等号は$\ket{\psi}$が基底状態のときに限る。そこでパラメータ$\boldsymbol{\theta}$を持つ量子回路で状態$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を最小化すれば$E_0$に近づく。これがVQEである。

- **アルゴリズムの流れ**

  1. 量子コンピュータで$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を測定する
  2. 古典コンピュータの最適化手法で$\boldsymbol{\theta}$を更新する
  3. $E(\boldsymbol{\theta})$が収束するまで1, 2を繰り返す

## 2. 例：$H = Z + X$

1量子ビットのハミルトニアン

\begin{equation}
H = Z + X = \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}
\end{equation}

を例に、VQEの各ステップを具体的に追う。固有値は$\pm\sqrt{2}$なので、厳密な基底エネルギーは$E_0 = -\sqrt{2} \approx -1.4142$である。

### 2.1 アンザッツ（試行状態）

$\ket{0}$に$R_y(\theta) = e^{-i\theta Y/2}$を1個作用させた状態を使う。

\begin{equation}
\ket{\psi(\theta)} = R_y(\theta)\ket{0} = \cos\frac{\theta}{2}\ket{0} + \sin\frac{\theta}{2}\ket{1}
\end{equation}

- **$R_y$を選ぶ理由**

  $H$は実対称行列なので、基底状態の振幅は（全体位相を除いて）実数にとれる。$R_y(\theta)$は$\ket{0}$から実振幅のすべての状態を1パラメータで作れる。ブロッホ球で見ると、$R_y(\theta)\ket{0}$のブロッホベクトルは$(\sin\theta, 0, \cos\theta)$で、x–z平面の大円上を動く。基底状態$-\frac{1}{\sqrt{2}}(1, 0, 1)$もこの平面上にあるので、必ず到達できる。

  一方、$R_x(\theta)\ket{0}$はy–z平面上を動くので$\braket{X} = 0$のままとなり、$E = \cos\theta \ge -1$となって$E_0$には届かない。固有値を知らなくても、ハミルトニアンの性質（ここでは実数性）からアンザッツを決められる。

このとき

\begin{align}
\braket{Z} &= \cos^2\frac{\theta}{2} - \sin^2\frac{\theta}{2} = \cos\theta \\
\braket{X} &= 2\cos\frac{\theta}{2}\sin\frac{\theta}{2} = \sin\theta
\end{align}

なので、エネルギーは

\begin{equation}
E(\theta) = \cos\theta + \sin\theta = \sqrt{2}\sin\left(\theta + \frac{\pi}{4}\right)
\end{equation}

となる。最小値は$\theta^* = -3\pi/4$のときの$-\sqrt{2}$で、$E_0$と一致する。

### 2.2 エネルギーの測定とショット雑音

量子コンピュータで直接測れるのは計算基底（$Z$基底）での0/1だけである。そこで$H$をパウリ演算子の項に分け、項ごとに測定する。

\begin{equation}
E(\theta) = \braket{Z} + \braket{X}
\end{equation}

- **$\braket{Z}$の測定**：そのまま$Z$基底で測定する。$N$回のうち$0$が$n_0$回、$1$が$n_1$回出たら

  \begin{equation}
  \braket{Z} \approx \frac{n_0 - n_1}{N}
  \end{equation}

- **$\braket{X}$の測定**：測定の直前にアダマールゲート$H_{\mathrm{gate}}$をかけてから$Z$基底で測る。$H_{\mathrm{gate}}^\dagger Z H_{\mathrm{gate}} = X$なので、これは$X$の測定と同じである。

- **ショット雑音**

  1回の測定結果は$\pm1$の確率変数なので、$N$ショットの平均には統計誤差がある。$\braket{Z}$の推定値の分散は$(1 - \braket{Z}^2)/N$である（$X$も同様）。$Z$と$X$を別々の$N$ショットで測ると、エネルギー推定値の標準偏差は

  \begin{equation}
  \delta = \sqrt{\frac{1 - \braket{Z}^2}{N} + \frac{1 - \braket{X}^2}{N}} \le \sqrt{\frac{2}{N}}
  \end{equation}

  となる。例えば$\theta = \pi/2$（状態$\ket{+}$）、$N = 1000$では$\braket{Z} = 0$、$\braket{X} = 1$なので、$\delta = 1/\sqrt{1000} \approx 0.03$である。誤差は$1/\sqrt{N}$でしか減らないので、精度を10倍にするにはショット数が100倍必要になる。

### 2.3 勾配の計算：パラメータシフト則

勾配降下法$\theta \leftarrow \theta - \eta\, \frac{dE}{d\theta}$には勾配が必要である。量子コンピュータから得られるのは「ある$\theta$でのエネルギーの推定値」だけなので、勾配もエネルギーの測定から求める必要がある。

- **主張**

  パウリ演算子$G$（$G^2 = I$）を生成子とするゲート$U(\theta) = e^{-i\theta G/2}$について

  \begin{equation}
  \frac{dE}{d\theta} = \frac{E(\theta + \pi/2) - E(\theta - \pi/2)}{2}
  \end{equation}

  が厳密に成り立つ。

- **補題**（エネルギーの正弦波性）

  パウリ演算子$G$（$G^\dagger = G$、$G^2 = I$）を生成子とする回転ゲート$U(\theta) = e^{-i\theta G/2}$をパラメータ$\theta$で含む回路について、他のパラメータを固定すると、定数$A, B, C$を用いて

  \begin{equation}
  E(\theta) = A + B\cos\theta + C\sin\theta
  \end{equation}

  と書ける。

  **証明** 回路を「ゲートの前の部分$V$」「注目するゲート$U(\theta)$」「ゲートの後ろの部分$W$」に分ける。$\ket{\varphi} = V\ket{0}$とおき、後ろの回路を観測量の側に吸収して$O = W^\dagger H W$とおくと

  \begin{equation}
  E(\theta) = \bra{0} V^\dagger U^\dagger(\theta) W^\dagger H W U(\theta) V \ket{0}
  = \bra{\varphi} U^\dagger(\theta)\, O\, U(\theta) \ket{\varphi}
  \end{equation}

  と書ける。$\ket{\varphi}$と$O$は$\theta$を含まない。本例では前後にゲートがないので、$\ket{\varphi} = \ket{0}$、$G = Y$、$O = H = Z + X$である。

  $G^2 = I$より、指数関数のテイラー展開を偶数次と奇数次に分けると

  \begin{equation}
  U(\theta) = \cos\frac{\theta}{2}\, I - i\sin\frac{\theta}{2}\, G = c\,I - i s\,G
  \end{equation}

  （$c = \cos\frac{\theta}{2}$、$s = \sin\frac{\theta}{2}$）となる。$U^\dagger(\theta) = c\,I + i s\,G$なので

  \begin{equation}
  U^\dagger O U = (c + isG)\, O\, (c - isG) = c^2\, O + s^2\, GOG + i cs\,(GO - OG)
  \end{equation}

  となる。$c^2 = \frac{1 + \cos\theta}{2}$、$s^2 = \frac{1 - \cos\theta}{2}$、$cs = \frac{\sin\theta}{2}$を代入して$\ket{\varphi}$で期待値をとると

  \begin{equation}
  E(\theta) = A + B\cos\theta + C\sin\theta
  \end{equation}

  \begin{equation}
  A = \frac{\braket{O} + \braket{GOG}}{2}, \quad
  B = \frac{\braket{O} - \braket{GOG}}{2}, \quad
  C = \frac{\braket{i[G, O]}}{2}
  \end{equation}

  を得る（期待値は$\ket{\varphi}$に関するもの）。$A, B, C$は$\theta$によらない。$\square$

- **定理**（任意のシフト幅での厳密な勾配）

  補題の条件の下で、$\sin h \neq 0$を満たす任意の$h$について

  \begin{equation}
  \frac{dE}{d\theta} = \frac{E(\theta + h) - E(\theta - h)}{2\sin h}
  \end{equation}

  が厳密に成り立つ。

  **証明** 補題より$E(\theta) = A + B\cos\theta + C\sin\theta$なので

  \begin{equation}
  \frac{dE}{d\theta} = -B\sin\theta + C\cos\theta
  \end{equation}

  である。一方、加法定理より

  \begin{align}
  \cos(\theta + h) - \cos(\theta - h) &= -2\sin\theta\sin h \\
  \sin(\theta + h) - \sin(\theta - h) &= 2\cos\theta\sin h
  \end{align}

  なので、定数$A$は打ち消し合い

  \begin{equation}
  E(\theta + h) - E(\theta - h) = 2\sin h\,(-B\sin\theta + C\cos\theta) = 2\sin h\, \frac{dE}{d\theta}
  \end{equation}

  となる。$\sin h \neq 0$なら両辺を$2\sin h$で割れる。$\square$

  通常の中心差分$\frac{E(\theta+h) - E(\theta-h)}{2h}$は、分母$2h$が$2\sin h$と近似的に等しい$h \to 0$でしか正しくない。一方この定理は、$E$が正弦波であることを使っているので、有限の$h$でも近似誤差がない。

- **系**（パラメータシフト則）

  定理で$h = \pi/2$（$\sin h = 1$）とすると

  \begin{equation}
  \frac{dE}{d\theta} = \frac{E(\theta + \pi/2) - E(\theta - \pi/2)}{2}
  \end{equation}

  を得る。$\theta$を$\pm\pi/2$ずらした2つの回路でエネルギーを測定すれば、厳密な勾配が求まる。$h = \pi/2$を選ぶのは、ショット雑音による勾配の誤差$\frac{\delta}{\sqrt{2}\sin h}$が最小になるからである（2.4節）。

- **本例での確認**

  $\braket{O} = \bra{0}(Z + X)\ket{0} = 1$。$GOG = Y(Z + X)Y = -Z - X$より$\braket{GOG} = -1$。$i[Y, Z + X] = 2Z - 2X$より$\braket{i[G, O]} = 2$。よって$A = 0$、$B = 1$、$C = 1$となり、$E(\theta) = \cos\theta + \sin\theta$と一致する。

  $\theta = 0$では$E(\pi/2) = 1$、$E(-\pi/2) = -1$より勾配は$(1 - (-1))/2 = 1$となり、$E'(0) = -\sin 0 + \cos 0 = 1$と一致する。

### 2.4 なぜ$\pi/2$ずらすのか：ショット雑音との関係

$E(\theta)$が正弦波であれば、任意の$h$（$\sin h \neq 0$）について

\begin{equation}
\frac{dE}{d\theta} = \frac{E(\theta + h) - E(\theta - h)}{2\sin h}
\end{equation}

が厳密に成り立つ。雑音がなければ$h$は何でもよいが、実際には各推定値に標準偏差$\delta$のショット雑音が乗る。

- 分子の信号は$2\sin h \cdot E'(\theta)$で、$\sin h$に比例する
- 分子の雑音は独立な2つの推定値の差なので$\sqrt{2}\,\delta$で、$h$によらない

したがって勾配の推定誤差は

\begin{equation}
\sigma_{\mathrm{grad}} = \frac{\sqrt{2}\,\delta}{2\sin h} = \frac{\delta}{\sqrt{2}\sin h}
\end{equation}

となり、$\sin h = 1$、つまり$h = \pi/2$で最小になる。これがパラメータシフト則のシフト量である。

$\delta = 0.03$での比較：

| 方法 | $h$ | 勾配の誤差 | 同じ精度に必要なショット数 |
|---|---|---|---|
| 数値微分（中心差分） | $0.01$ | 約$2.1$ | 約$10^4$倍 |
| シフト量を小さくした場合 | $\pi/6$ | 約$0.042$ | $4$倍 |
| パラメータシフト則 | $\pi/2$ | 約$0.021$ | $1$倍（基準） |

通常の数値微分は$h$を小さくしないと近似誤差$O(h^2)$が残るが、小さい$h$で割ると雑音が$1/h$で拡大される。パラメータシフト則は大きくずらしても厳密なので、このトレードオフがない。

### 2.5 最適化の実行例

$\theta_0 = 0$、学習率$\eta = 0.5$として、パラメータシフト則で求めた勾配で勾配降下法を行う（雑音なしの値）。

| 反復 | $\theta$ | $E(\theta)$ | $dE/d\theta$ |
|---|---|---|---|
| 0 | $0.000$ | $1.000$ | $1.000$ |
| 1 | $-0.500$ | $0.398$ | $1.357$ |
| 2 | $-1.179$ | $-0.542$ | $1.306$ |
| 3 | $-1.832$ | $-1.224$ | $0.708$ |
| 4 | $-2.186$ | $-1.394$ | $0.240$ |
| 5 | $-2.306$ | $-1.412$ | $0.071$ |
| 6 | $-2.341$ | $-1.4141$ | $0.021$ |

$E(\theta)$は単調に減少し、厳密解$E_0 = -\sqrt{2} \approx -1.4142$（$\theta^* = -3\pi/4 \approx -2.356$）に収束する。実機ではここに2.2節のショット雑音が加わるため、最終的なエネルギーは$\delta$程度の揺らぎを持つ。

## 2. エネルギーの測定

ハミルトニアンをPauli演算子の積の和$H = \sum_a h_a P_a$で書くと、$E(\boldsymbol{\theta}) = \sum_a h_a\langle P_a\rangle$である。各$\langle P_a\rangle$は、Trotter分解のノートブック（[03_trotter.ipynb](03_trotter.ipynb)の4節）と同じ基底変換をしてから測定すれば求まる。同時に対角化できる（同じ基底で測れる）項をまとめて測ると、測定の種類を減らせる。

## 3. アンザッツ（回路の形）

パラメータ付きの回路$\ket{\psi(\boldsymbol{\theta})}$の形をアンザッツと呼ぶ。どう選ぶかで結果が大きく変わる。

- **Hardware-efficient ansatz**：実機で実行しやすい1量子ビット回転と、隣どうしのエンタングルゲートを交互に重ねる。汎用的だが、物理的な構造を使わないので、必要なパラメータが多くなりやすい。
- **Hamiltonian Variational Ansatz（HVA）**：$H = \sum_k H_k$の各部分の指数関数$e^{-i\theta_k H_k}$を交互に重ねる。Trotter分解の1ステップで、回転角を自由なパラメータにしたものと同じ形である。量子最適化のQAOAも同じ構造を持つ。ハミルトニアンの対称性を自然に保つ。
- **UCC（unitary coupled cluster）**：量子化学で使われる、電子の励起の形をもとにしたアンザッツ。

表現力が足りないと厳密な基底状態に届かず、逆にパラメータが多すぎると最適化が難しくなる。

## 4. 最適化と注意点

- **最適化手法**：勾配を使わない方法（COBYLA、Nelder-Meadなど）と、勾配を使う方法（BFGS、Adamなど）がある。実機では、回路のパラメータを$\pm\pi/2$ずらした2回の測定から勾配を求める**パラメータシフト則**が使える。
- **局所解**：エネルギーの地形には局所的な最小値が多く、初期値によって結果が変わる。層を1つずつ増やし、前の層の最適値を初期値に使う（ウォームスタート）と安定しやすい。
- **Barren plateau**：量子ビット数や回路の深さが大きいと、ランダムな初期値では勾配が指数関数的に小さくなり、最適化が進まなくなることがある。
- **変分原理による保証**：得られるエネルギーは必ず$E_0$以上なので、値が小さいほど良い近似である。ただし、どれだけ$E_0$に近いかはVQEだけではわからない。厳密解など他の手法と比べる必要がある。

---
**前へ**: [03 Trotter分解](03_trotter.ipynb)　｜　**次へ**: [05 TEBD](05_tebd.ipynb)